In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
raw.samples <- read_tsv('/content/data/brca_metabric/data_clinical_sample.txt', comment = '#')
raw.mutations <- read_tsv('/content/data/brca_metabric/data_mutations.txt', comment = '#')

Rows: 2509 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): PATIENT_ID, SAMPLE_ID, CANCER_TYPE, CANCER_TYPE_DETAILED, ER_STATUS...
dbl (4): GRADE, TUMOR_SIZE, TUMOR_STAGE, TMB_NONSYNONYMOUS

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 17272 Columns: 45
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (19): Hugo_Symbol, Center, NCBI_Build, Chromosome, Strand, Consequence, ...
dbl  (4): Start_Position, End_Position, Protein_position, Hotspot
lgl (22): Entrez_Gene_Id, dbSNP_RS, dbSNP_Val_Status, Matched_Norm_Sample_Ba...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Samples

In [4]:
head(raw.samples)

PATIENT_ID,SAMPLE_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,ER_STATUS,HER2_STATUS,GRADE,ONCOTREE_CODE,PR_STATUS,SAMPLE_TYPE,TUMOR_SIZE,TUMOR_STAGE,TMB_NONSYNONYMOUS
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
MB-0000,MB-0000,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,3,IDC,Negative,Primary,22,2,0.000000
MB-0002,MB-0002,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,3,IDC,Positive,Primary,10,1,2.615035
MB-0005,MB-0005,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2,IDC,Positive,Primary,15,2,2.615035
MB-0006,MB-0006,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,2,MDLC,Positive,Primary,25,2,1.307518
MB-0008,MB-0008,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,3,MDLC,Positive,Primary,40,2,2.615035
MB-0010,MB-0010,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,3,IDC,Positive,Primary,31,4,5.230071


In [5]:
summary(raw.samples)

  PATIENT_ID         SAMPLE_ID         CANCER_TYPE        CANCER_TYPE_DETAILED
 Length:2509        Length:2509        Length:2509        Length:2509         
 Class :character   Class :character   Class :character   Class :character    
 Mode  :character   Mode  :character   Mode  :character   Mode  :character    
                                                                              
                                                                              
                                                                              
                                                                              
  ER_STATUS         HER2_STATUS            GRADE       ONCOTREE_CODE     
 Length:2509        Length:2509        Min.   :1.000   Length:2509       
 Class :character   Class :character   1st Qu.:2.000   Class :character  
 Mode  :character   Mode  :character   Median :3.000   Mode  :character  
                                       Mean   :2.412                    

In [6]:
glimpse(raw.samples)

Rows: 2,509
Columns: 13
$ PATIENT_ID           <chr> "MB-0000", "MB-0002", "MB-0005", "MB-0006", "MB-0…
$ SAMPLE_ID            <chr> "MB-0000", "MB-0002", "MB-0005", "MB-0006", "MB-0…
$ CANCER_TYPE          <chr> "Breast Cancer", "Breast Cancer", "Breast Cancer"…
$ CANCER_TYPE_DETAILED <chr> "Breast Invasive Ductal Carcinoma", "Breast Invas…
$ ER_STATUS            <chr> "Positive", "Positive", "Positive", "Positive", "…
$ HER2_STATUS          <chr> "Negative", "Negative", "Negative", "Negative", "…
$ GRADE                <dbl> 3, 3, 2, 2, 3, 3, 2, 3, 2, 3, 3, 2, 2, 1, 3, 3, 2…
$ ONCOTREE_CODE        <chr> "IDC", "IDC", "IDC", "MDLC", "MDLC", "IDC", "IDC"…
$ PR_STATUS            <chr> "Negative", "Positive", "Positive", "Positive", "…
$ SAMPLE_TYPE          <chr> "Primary", "Primary", "Primary", "Primary", "Prim…
$ TUMOR_SIZE           <dbl> 22, 10, 15, 25, 40, 31, 10, 65, 29, 34, 16, 28, 2…
$ TUMOR_STAGE          <dbl> 2, 1, 2, 2, 2, 4, 2, 3, 2, 2, 2, 2, 4, 1, 2, 2, 2…
$ TMB_NONSYNONYM

In [10]:
names(raw.samples)

[1] "PATIENT_ID"           "SAMPLE_ID"            "CANCER_TYPE"         
 [4] "CANCER_TYPE_DETAILED" "ER_STATUS"            "HER2_STATUS"         
 [7] "GRADE"                "ONCOTREE_CODE"        "PR_STATUS"           
[10] "SAMPLE_TYPE"          "TUMOR_SIZE"           "TUMOR_STAGE"         
[13] "TMB_NONSYNONYMOUS"

## Mutations

In [7]:
head(raw.mutations)

Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,⋯,n_ref_count,n_alt_count,HGVSc,HGVSp,HGVSp_Short,Transcript_ID,RefSeq,Protein_position,Codons,Hotspot
<chr>,<lgl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>
TP53,NA,METABRIC,GRCh37,17,7579344,7579345,+,frameshift_variant,Frame_Shift_Ins,⋯,NA,NA,ENST00000269305.4:c.343dup,p.His115ProfsTer34,p.H115Pfs*34,ENST00000269305,NM_001126112.2,114,-/C,0
TP53,NA,METABRIC,GRCh37,17,7579346,7579347,+,protein_altering_variant,In_Frame_Ins,⋯,NA,NA,ENST00000269305.4:c.340_341insCTG,p.Leu114delinsSerVal,p.L114delinsSV,ENST00000269305,NM_001126112.2,114,ttg/tCTGtg,0
MLLT4,NA,METABRIC,GRCh37,6,168299111,168299111,+,missense_variant,Missense_Mutation,⋯,NA,NA,ENST00000392108.3:c.1544G>T,p.Gly515Val,p.G515V,ENST00000392108,NM_001040000.2,515,gGa/gTa,0
NF2,NA,METABRIC,GRCh37,22,29999995,29999995,+,missense_variant,Missense_Mutation,⋯,NA,NA,ENST00000338641.4:c.8G>T,p.Gly3Val,p.G3V,ENST00000338641,NM_000268.3,3,gGg/gTg,0
SF3B1,NA,METABRIC,GRCh37,2,198288682,198288682,+,synonymous_variant,Silent,⋯,NA,NA,ENST00000335508.6:c.45T>A,p.Ile15=,p.I15=,ENST00000335508,NM_012433.2,15,atT/atA,0
NT5E,NA,METABRIC,GRCh37,6,86195125,86195125,+,synonymous_variant,Silent,⋯,NA,NA,ENST00000257770.3:c.924T>C,p.Ile308=,p.I308=,ENST00000257770,NM_002526.3,308,atT/atC,0


In [8]:
summary(raw.mutations)

 Hugo_Symbol        Entrez_Gene_Id    Center           NCBI_Build       
 Length:17272       Mode:logical   Length:17272       Length:17272      
 Class :character   NA's:17272     Class :character   Class :character  
 Mode  :character                  Mode  :character   Mode  :character  
                                                                        
                                                                        
                                                                        
                                                                        
  Chromosome        Start_Position       End_Position          Strand         
 Length:17272       Min.   :   193227   Min.   :   193227   Length:17272      
 Class :character   1st Qu.: 28443610   1st Qu.: 28443611   Class :character  
 Mode  :character   Median : 65339122   Median : 65339122   Mode  :character  
                    Mean   : 83135984   Mean   : 83135984                     
                    3

In [9]:
glimpse(raw.mutations)

Rows: 17,272
Columns: 45
$ Hugo_Symbol                   <chr> "TP53", "TP53", "MLLT4", "NF2", "SF3B1",…
$ Entrez_Gene_Id                <lgl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ Center                        <chr> "METABRIC", "METABRIC", "METABRIC", "MET…
$ NCBI_Build                    <chr> "GRCh37", "GRCh37", "GRCh37", "GRCh37", …
$ Chromosome                    <chr> "17", "17", "6", "22", "2", "6", "7", "1…
$ Start_Position                <dbl> 7579344, 7579346, 168299111, 29999995, 1…
$ End_Position                  <dbl> 7579345, 7579347, 168299111, 29999995, 1…
$ Strand                        <chr> "+", "+", "+", "+", "+", "+", "+", "+", …
$ Consequence                   <chr> "frameshift_variant", "protein_altering_…
$ Variant_Classification        <chr> "Frame_Shift_Ins", "In_Frame_Ins", "Miss…
$ Variant_Type                  <chr> "INS", "INS", "SNP", "SNP", "SNP", "SNP"…
$ Reference_Allele              <chr> "-", "-", "G", "G", "A", "T", "C", "C", …
$ Tumor_Seq_All

In [11]:
names(raw.mutations)

[1] "Hugo_Symbol"                   "Entrez_Gene_Id"               
 [3] "Center"                        "NCBI_Build"                   
 [5] "Chromosome"                    "Start_Position"               
 [7] "End_Position"                  "Strand"                       
 [9] "Consequence"                   "Variant_Classification"       
[11] "Variant_Type"                  "Reference_Allele"             
[13] "Tumor_Seq_Allele1"             "Tumor_Seq_Allele2"            
[15] "dbSNP_RS"                      "dbSNP_Val_Status"             
[17] "Tumor_Sample_Barcode"          "Matched_Norm_Sample_Barcode"  
[19] "Match_Norm_Seq_Allele1"        "Match_Norm_Seq_Allele2"       
[21] "Tumor_Validation_Allele1"      "Tumor_Validation_Allele2"     
[23] "Match_Norm_Validation_Allele1" "Match_Norm_Validation_Allele2"
[25] "Verification_Status"           "Validation_Status"            
[27] "Mutation_Status"               "Sequencing_Phase"             
[29] "Sequence_Source"               "Validation_Method"            
[31] "Score"                         "BAM_File"                     
[33] "Sequencer"                     "t_ref_count"                  
[35] "t_alt_count"                   "n_ref_count"                  
[37] "n_alt_count"                   "HGVSc"                        
[39] "HGVSp"                         "HGVSp_Short"                  
[41] "Transcript_ID"                 "RefSeq"                       
[43] "Protein_position"              "Codons"                       
[45] "Hotspot"